![](https://media.bizj.us/view/img/10530430/gettyimages-172241300*1200xx724-407-0-38.jpg)

# Business Problem

- Can a machine learning project be realized for salary predictions of baseball players whose salary information and career statistics from 1986 are shared?

# Dataset Story

- This data set was originally taken from the StatLib library at Carnegie Mellon University.

- The data set is part of the data used in the 1988 ASA Graphics Division Poster Session.

- Salary data was originally taken from Sports Illustrated, April 20, 1987.

- 1986 and career statistics are from Collier Books, published by Macmillan Publishing Company, New York Derived from the 1987 Baseball Encyclopedia Update.

**Variables**

- **AtBat:** The number of times a baseball was hit by a bat during the 1986-1987 season.

- **Hits:** Number of hits in the 1986-1987 season.

- **HmRun:** Most valuable at-bats in the 1986-1987 season.

- **Runs:** Runs scored for his team during the 1986-1987 season.

- **RBI:** The number of times a batter scored a run while batting.

- **Walks:** The number of errors committed by an opposing batter.

- **Years:** Number of years (years) a player has played in the major leagues.

- **CAtBats:** Number of at-bats during the player's career.

- **CHits:** Number of hits during the player's career.

- **CHmRun:** Player's most valuable run during his career.

- **CRuns:** The number of runs scored for the team during the player's career.

- **CRBI:** Number of runs scored by the player during his career.

- **CWalks:** The number of errors the player has allowed to opposing players during his career.

- **League:** A factor with A and N levels indicating the league the player played in until the end of the season.

- **Division:** A factor with levels E and W that indicates the position the player played at the end of 1986.

- **PutOuts:** Helping your teammate in the game.

- **Assits:** The number of assists made by the player in the 1986-1987 season.

- **Errors:** Number of errors by the player in the 1986-1987 season.

- **Salary:** Player's salary for the 1986-1987 season (in thousands).

- **NewLeague:** A factor with A and N levels indicating the player's league at the beginning of the 1987 season.

# Road Map

- **1. Import Required Libraries**

- **2. Adjusting Row Column Settings**

- **3. Loading the data Set**

- **4. Exploratory Data Analysis**

- **5. Capturing / Detecting Numeric and Categorical Variables**

- **6. Analysis of Categorical Variables**

- **7. Analysis of Numerical Variables**

- **8. Analysis of Categorical Variables by Target**

- **9. Analysis of Numeric Variables by Target**

- **10. Correlation Analysis**

- **11. Distribution of the Dependent Variable**

- **12. Examining the Logarithm of the Dependent Variable**

- **13. Outliers Analysis**

- **14. Missing Value Analysis**

- **15. Rare Analysis**

- **16. Feature Extraction**

- **17. Encoding**

- **18. Standardization Process**

- **19. Creating Model**

- **20. Hyperparameter optimization**

- **21. Final Model Predictions and Comparison with True Prices**

# 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn import metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score,GridSearchCV
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, StandardScaler, RobustScaler


warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter("ignore", category=ConvergenceWarning)

# 2. Adjusting Row Column Settings

In [ ]:
pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

# 3. Loading the data Set

In [ ]:
df = pd.read_csv("/kaggle/input/hitters-baseball-data/Hitters.csv")

In [ ]:
df.head()

# 4. Exploratory Data Analysis

In [ ]:
# Preliminary examination of the data set

def check_df(dataframe, head=5):
    print('##################### Shape #####################')
    print(dataframe.shape)
    print('##################### Types #####################')
    print(dataframe.dtypes)
    print('##################### Head #####################')
    print(dataframe.head(head))
    print('##################### Tail #####################')
    print(dataframe.tail(head))
    print('##################### NA #####################')
    print(dataframe.isnull().sum())
    print('##################### Quantiles #####################')
    print(dataframe.describe([0, 0.05, 0.50, 0.95, 0.99, 1]).T)

In [ ]:
check_df(df)

# 5. Capturing / Detecting Numeric and Categorical Variables

In [ ]:
def grab_col_names(dataframe, cat_th=10, car_th=20):
    """

    Returns the names of categorical, numeric and categorical but cardinal variables in the data set.
    Note Categorical variables include categorical variables with numeric appearance.

    Parameters
    ------
        dataframe: dataframe
                Variable names of the dataframe to be taken
        cat_th: int, optional
                class threshold for numeric but categorical variables
        car_th: int, optinal
                class threshold for categorical but cardinal variables

    Returns
    ------
        cat_cols: list
                Categorical variable list
        num_cols: list
                Numeric variable list
        cat_but_car: list
                List of cardinal variables with categorical appearance

    Examples
    ------
        import seaborn as sns
        df = sns.load_dataset("iris")
        print(grab_col_names(df))


    Notes
    ------
        cat_cols + num_cols + cat_but_car = total number of variables
        num_but_cat is inside cat_cols.
        The sum of the 3 return lists equals the total number of variables: cat_cols + num_cols + cat_but_car = number of variables

    """

    # cat_cols, cat_but_car
    cat_cols = [col for col in dataframe.columns if dataframe[col].dtypes == "O"] 

    num_but_cat = [col for col in dataframe.columns if dataframe[col].nunique() < cat_th and
                   dataframe[col].dtypes != "O"]

    cat_but_car = [col for col in dataframe.columns if dataframe[col].nunique() > car_th and
                   dataframe[col].dtypes == "O"]

    cat_cols = cat_cols + num_but_cat

    cat_cols = [col for col in cat_cols if col not in cat_but_car] 

    num_cols = [col for col in dataframe.columns if dataframe[col].dtypes != "O"] 

    num_cols = [col for col in num_cols if col not in num_but_cat] 
    
    print(f"Observations: {dataframe.shape[0]}") 
    print(f"Variables: {dataframe.shape[1]}") 
    print(f'cat_cols: {len(cat_cols)}') 
    print(f'num_cols: {len(num_cols)}') 
    print(f'cat_but_car: {len(cat_but_car)}') 
    print(f'num_but_cat: {len(num_but_cat)}') 


    return cat_cols, num_cols, cat_but_car, num_but_cat

In [ ]:
cat_cols, num_cols, cat_but_car,  num_but_cat = grab_col_names(df)

In [ ]:
df.head()

In [ ]:
cat_cols

In [ ]:
num_cols

In [ ]:
cat_but_car

In [ ]:
num_but_cat

# 6. Analysis of Categorical Variables

In [ ]:
def cat_summary(dataframe, col_name, plot=False):
    print(pd.DataFrame({col_name: dataframe[col_name].value_counts(),
                        'Ratio': 100 * dataframe[col_name].value_counts() / len(dataframe)}))
    print('##########################################')
    if plot:
        plt.figure(figsize=(12,6))
        sns.countplot(x=dataframe[col_name], data=dataframe)
        plt.show(block=True)

In [ ]:
for col in cat_cols:
    cat_summary(df, col, plot=True)

# 7. Analysis of Numerical Variables

In [ ]:
def num_summary(dataframe, numerical_col, plot=False):
    quantiles = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]
    print(dataframe[numerical_col].describe(quantiles).T)

    if plot:
        dataframe[numerical_col].hist(bins=20)
        
        plt.xlabel(numerical_col)
        plt.title(numerical_col)
        plt.show(block=True)

In [ ]:
for col in num_cols:
    num_summary(df, col, plot=True)

# 8. Analysis of Categorical Variables by Target

In [ ]:
def target_summary_with_cat(dataframe, target, categorical_col, plot=False):
    print(pd.DataFrame({'TARGET_MEAN': dataframe.groupby(categorical_col)[target].mean()}), end='\n\n\n')
    if plot:
        sns.barplot(x=categorical_col, y=target, data=dataframe)
        plt.show(block=True)

In [ ]:
for col in cat_cols:
    target_summary_with_cat(df, 'Salary', col, plot=True)

# 9. Analysis of Numeric Variables by Target

In [ ]:
def target_summary_with_num(dataframe, target, numerical_col, plot=False):
    print(pd.DataFrame({numerical_col+'_mean': dataframe.groupby(target)[numerical_col].mean()}), end='\n\n\n')
    if plot:
        sns.barplot(x=target, y=numerical_col, data=dataframe)
        plt.show(block=True)

In [ ]:
for col in num_cols:
    target_summary_with_cat(df, 'Salary', col, plot=False)

# 10. Analysis of Correlation

In [ ]:
def high_correlated_cols(dataframe, plot=False, corr_th=0.70):
    corr = dataframe.corr()
    cor_matrix = corr.abs()
    upper_triangle_matrix = cor_matrix.where(np.triu(np.ones(cor_matrix.shape), k=1).astype(np.bool))
    drop_list = [col for col in upper_triangle_matrix.columns if any(upper_triangle_matrix[col] > corr_th)]
    if plot:
        import seaborn as sns
        import matplotlib.pyplot as plt
        sns.set(rc={'figure.figsize': (16, 14)})
        sns.heatmap(corr, cmap="RdBu", annot=True, fmt=".2f")  # annot=True added here
        plt.show()
    return drop_list

In [ ]:
high_correlated_cols(df, plot=True)

In [ ]:
corr = df[num_cols].corr()

In [ ]:
corr

# 11. Distribution of the Dependent Variable

In [ ]:
df["Salary"].hist(bins=100)
plt.show(block=True)

# 12. Examining the Logarithm of the Dependent Variable

In [ ]:
np.log1p(df['Salary']).hist(bins=50)
plt.show(block=True)

# 13. Outliers Analysis

In [ ]:
def outlier_thresholds(dataframe, col_name, q1=0.25, q3=0.75):
    quartile1 = dataframe[col_name].quantile(q1)
    quartile3 = dataframe[col_name].quantile(q3)
    interquantile_range = quartile3 - quartile1
    up_limit = quartile3 + 1.5 * interquantile_range
    low_limit = quartile1 - 1.5 * interquantile_range
    return low_limit, up_limit

In [ ]:
def check_outlier(dataframe, col_name):
    low_limit, up_limit = outlier_thresholds(dataframe, col_name)
    if dataframe[(dataframe[col_name] > up_limit) | (dataframe[col_name] < low_limit)].any(axis=None):
        return True
    else:
        return False

In [ ]:
def replace_with_thresholds(dataframe, variable):
    low_limit, up_limit = outlier_thresholds(dataframe, variable)
    dataframe.loc[(dataframe[variable] < low_limit), variable] = low_limit
    dataframe.loc[(dataframe[variable] > up_limit), variable] = up_limit

In [ ]:
for col in num_cols:
    print(col, check_outlier(df, col))

In [ ]:
for col in num_cols:
    if check_outlier(df, col):
        replace_with_thresholds(df, col)

In [ ]:
for col in num_cols:
    print(col, check_outlier(df, col))

# 14. Missing Value Analysis

In [ ]:
def missing_values_table(dataframe, na_name=False):
    na_columns = [col for col in dataframe.columns if dataframe[col].isnull().sum() > 0]

    n_miss = dataframe[na_columns].isnull().sum().sort_values(ascending=False)

    ratio = (dataframe[na_columns].isnull().sum() / dataframe.shape[0] * 100).sort_values(ascending=False)

    missing_df = pd.concat([n_miss, np.round(ratio, 2)], axis=1, keys=['n_miss', 'ratio'])

    print(missing_df, end="\n")

    if na_name:
        return na_columns

In [ ]:
missing_values_table(df)

In [ ]:
# Fill in missing variables by League and Division (A,E)(A,W)(N,E)(N,W)

df.loc[(df["Salary"].isnull()) & (df["League"] == "A") & (df["Division"] == "E"), "Salary"] = df.groupby(["League","Division"])["Salary"].mean()["A","E"]

df.loc[(df["Salary"].isnull()) & (df["League"] == "A") & (df["Division"] == "W"), "Salary"] = df.groupby(["League","Division"])["Salary"].mean()["A","W"]

df.loc[(df["Salary"].isnull()) & (df["League"] == "N") & (df["Division"] == "E"), "Salary"] = df.groupby(["League","Division"])["Salary"].mean()["N","E"]

df.loc[(df["Salary"].isnull()) & (df["League"] == "N") & (df["Division"] == "W"), "Salary"] = df.groupby(["League","Division"])["Salary"].mean()["N","W"]

In [ ]:
missing_values_table(df)

# 15. Rare Analysis

In [ ]:
def rare_analyser(dataframe, target, cat_cols):
    for col in cat_cols:
        print(col, ':', len(dataframe[col].value_counts()))
        print(pd.DataFrame({'COUNT': dataframe[col].value_counts(),
                            'RATIO': dataframe[col].value_counts() / len(dataframe),
                            'TARGET_MEAN': dataframe.groupby(col)[target].mean()}), end='\n\n\n')

In [ ]:
rare_analyser(df, "Salary", cat_cols)

In [ ]:
def rare_encoder(dataframe, rare_perc):
    temp_df = dataframe.copy()

    rare_columns = [col for col in temp_df.columns if temp_df[col].dtypes == 'O'
                    and (temp_df[col].value_counts() / len(temp_df) < rare_perc).any(axis=None)]

    for var in rare_columns:
        tmp = temp_df[var].value_counts() / len(temp_df)
        rare_labels = tmp[tmp < rare_perc].index
        temp_df[var] = np.where(temp_df[var].isin(rare_labels), 'Rare', temp_df[var])
    return temp_df

In [ ]:
rare_encoder(df, 0.01)

# 16. Feature Extraction

In [ ]:
new_num_cols=[col for col in num_cols if col!="Salary"]

In [ ]:
new_num_cols

In [ ]:
df[new_num_cols]=df[new_num_cols]+0.0000000001

In [ ]:
# "New Hit Ratio and Total Hits": Ratio of a player's seasonal hits to career hits plus the seasonal hits.
df['NEW_Hits'] = df['Hits'] / df['CHits'] + df['Hits']

# "New RBI Ratio": Ratio of a player's seasonal RBI to career RBI.
df['NEW_RBI'] = df['RBI'] / df['CRBI']

# "New Walks Ratio": Ratio of a player's seasonal walks to career walks.
df['NEW_Walks'] = df['Walks'] / df['CWalks']

# "Total PutOuts": A player's total putouts over the course of his career.
df['NEW_PutOuts'] = df['PutOuts'] * df['Years']

# "Hits Success Rate": A player's success rate in hits.
df["Hits_Success"] = (df["Hits"] / df["AtBat"]) * 100

# "Total Career RBI and AtBat": The product of a player's career RBI and career AtBat.
df["NEW_CRBI*CATBAT"] = df['CRBI'] * df['CAtBat']

# "Average Yearly Career Hits": A player's average career hits per year.
df["NEW_Chits"] = df["CHits"] / df["Years"]

# "Total Career HmRun": A player's total home runs over the course of his career.
df["NEW_CHmRun"] = df["CHmRun"] * df["Years"]

# "Average Yearly Career Runs": A player's average career runs per year.
df["NEW_CRuns"] = df["CRuns"] / df["Years"]

# "Total Career Hits": A player's total hits over the course of his career.
df["NEW_Chits"] = df["CHits"] * df["Years"]

# "RBI and Walks Product": The product of a player's RBI and Walks.
df["NEW_RW"] = df["RBI"] * df["Walks"]

# "RBI to Walks Ratio": The ratio of a player's RBI to Walks.
df["NEW_RBWALK"] = df["RBI"] / df["Walks"]

# "Career Hits to AtBat Ratio": The ratio of a player's career hits to career AtBats.
df["NEW_CH_CB"] = df["CHits"] / df["CAtBat"]

# "Career HmRun to AtBat Ratio": The ratio of a player's career home runs to career AtBats.
df["NEW_CHm_CAT"] = df["CHmRun"] / df["CAtBat"]

# "AtBat Difference": The difference between a player's yearly average career AtBats and seasonal AtBats.
df['NEW_Diff_Atbat'] = df['AtBat'] - (df['CAtBat'] / df['Years'])

# "Hits Difference": The difference between a player's yearly average career hits and seasonal hits.
df['NEW_Diff_Hits'] = df['Hits'] - (df['CHits'] / df['Years'])

# "HmRun Difference": The difference between a player's yearly average career home runs and seasonal home runs.
df['NEW_Diff_HmRun'] = df['HmRun'] - (df['CHmRun'] / df['Years'])

# "Runs Difference": The difference between a player's yearly average career runs and seasonal runs.
df['NEW_Diff_Runs'] = df['Runs'] - (df['CRuns'] / df['Years'])

# "RBI Difference": The difference between a player's yearly average career RBI and seasonal RBI.
df['NEW_Diff_RBI'] = df['RBI'] - (df['CRBI'] / df['Years'])

# "Walks Difference": The difference between a player's yearly average career walks and seasonal walks.
df['NEW_Diff_Walks'] = df['Walks'] - (df['CWalks'] / df['Years'])

In [ ]:
df.head()

# 17. Encoding

In [ ]:
df.info()

In [ ]:
cat_cols, num_cols, cat_but_car,  num_but_cat = grab_col_names(df)

In [ ]:
cat_cols

In [ ]:
num_cols

In [ ]:
cat_but_car

In [ ]:
num_but_cat

In [ ]:
def one_hot_encoder(dataframe, categorical_cols, drop_first=False):
    dataframe = pd.get_dummies(dataframe, columns=categorical_cols, drop_first=drop_first)
    return dataframe

In [ ]:
df = one_hot_encoder(df, cat_cols, drop_first=True)

In [ ]:
df.head()

# 18. Standardization Process

In [ ]:
num_cols = [col for col in num_cols if col not in ["Salary"]]

In [ ]:
scaler = RobustScaler()

In [ ]:
df[num_cols] = scaler.fit_transform(df[num_cols])

In [ ]:
df.head(10)

In [ ]:
# Editing of variable names.

df.columns = df.columns.str.replace(' ', '_')
df.columns = df.columns.str.replace('[^A-Za-z0-9_]+', '')
df.columns = df.columns.str.lower()

In [ ]:
df.head()

# 19. Creating Model

In [ ]:
y = df["salary"]

In [ ]:
X = df.drop(["salary"], axis=1)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=17)

In [ ]:
models = [('LR', LinearRegression()),
          ("Ridge", Ridge()),
          ("Lasso", Lasso()),
          ("ElasticNet", ElasticNet()),
          ('KNN', KNeighborsRegressor()),
          ('CART', DecisionTreeRegressor()),
          ('RF', RandomForestRegressor()),
          #('SVR', SVR()),
          ('GBM', GradientBoostingRegressor()),
          ("XGBoost", XGBRegressor(objective='reg:squarederror')),
          ("LightGBM", LGBMRegressor()),
          ("CatBoost", CatBoostRegressor(verbose=False))]

rmse_scores = []
r2_scores = []
mae_scores = []
mse_scores = []
execution_times = []

for name, regressor in models:
    start_time = time.time()

    # Fit the model
    regressor.fit(X_train, y_train)

    # Make predictions
    y_pred = regressor.predict(X_test)

    # Calculate RMSE
    rmse = np.mean(np.sqrt(-cross_val_score(regressor, X, y, cv=5, scoring="neg_mean_squared_error")))
    rmse_scores.append(rmse)
    
    # Calculate R^2 score
    r2 = metrics.r2_score(y_test, y_pred)
    r2_scores.append(r2)

    # Calculate MAE
    mae = metrics.mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)

    # Calculate MSE
    mse = metrics.mean_squared_error(y_test, y_pred)
    mse_scores.append(mse)

    # Calculate the execution time of the model
    execution_time = time.time() - start_time
    execution_times.append(execution_time)

    print(f"RMSE: {round(rmse, 4)} ({name})")
    print(f"R^2 Score: {round(r2, 4)} ({name})")
    print(f"MAE: {round(mae, 4)} ({name})")
    print(f"MSE: {round(mse, 4)} ({name})")
    print(f"Execution Time: {round(execution_time, 2)} seconds\n")

# 20. Hyperparameter optimization

In [ ]:
# Initialize the models
models = [('LR', LinearRegression()),
          ("Ridge", Ridge()),
          ("Lasso", Lasso()),
          ("ElasticNet", ElasticNet()),
          ('KNN', KNeighborsRegressor()),
          ('CART', DecisionTreeRegressor()),
          ('RF', RandomForestRegressor()),
          ('GBM', GradientBoostingRegressor()),
          ("XGBoost", XGBRegressor(objective='reg:squarederror')),
          ("LightGBM", LGBMRegressor()),
          ("CatBoost", CatBoostRegressor(verbose=False))]

# Initialize lists to store metrics
rmse_scores = []
r2_scores = []
mae_scores = []
mse_scores = []
execution_times = []

# Define the hyperparameters for each model

param_grids = {
    'LR': {},
    'Ridge': {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]},
    'Lasso': {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]},
    'ElasticNet': {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0], 'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]},
    'KNN': {'n_neighbors': [1, 3, 5, 7, 9]},
    'CART': {'max_depth': [None, 5, 10, 20, 30], 'min_samples_leaf': [1, 2, 3]},
    'RF': {'n_estimators': [10, 30, 50, 70, 100], 'max_depth': [None, 5, 10, 20]},
    'GBM': {'n_estimators': [10, 30, 50, 70, 100], 'learning_rate': [0.005, 0.01, 0.05, 0.1]},
    'XGBoost': {'n_estimators': [10, 30, 50, 70, 100], 'learning_rate': [0.005, 0.01, 0.05, 0.1]},
    'LightGBM': {'n_estimators': [10, 30, 50, 70, 100], 'learning_rate': [0.005, 0.01, 0.05, 0.1]},
    'CatBoost': {'iterations': [10, 30, 50, 70, 100], 'learning_rate': [0.005, 0.01, 0.05, 0.1], 'depth': [3, 4, 5, 6, 7]}
}

# Variables to store best model and score globally
best_global_score = np.inf
best_global_model = None

# Train and evaluate the models with hyperparameter tuning
for name, regressor in models:
    print(f"Hyperparameter Tuning for {name}:")
    start_time = time.time()

    if param_grids[name]:
        grid_search = GridSearchCV(regressor, param_grid=param_grids[name], cv=5, n_jobs=-1)
        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_

        print(f"Best parameters: {grid_search.best_params_}")
    else:
        best_model = regressor.fit(X_train, y_train)

    # Make predictions
    y_pred = best_model.predict(X_test)

    # Calculate RMSE
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    rmse_scores.append(rmse)

    # If current model's score is better, update best global model and score
    if rmse < best_global_score:
        best_global_score = rmse
        best_global_model = best_model

    # Calculate R^2 score
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)

    # Calculate MAE
    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)

    # Calculate MSE
    mse = mean_squared_error(y_test, y_pred)
    mse_scores.append(mse)

    # Calculate the execution time of the model
    execution_time = time.time() - start_time
    execution_times.append(execution_time)

    print(f"RMSE: {round(rmse, 4)} ({name})")
    print(f"R^2 Score: {round(r2, 4)} ({name})")
    print(f"MAE: {round(mae, 4)} ({name})")
    print(f"MSE: {round(mse, 4)} ({name})")
    print(f"Execution Time: {round(execution_time, 2)} seconds\n")

# 21. Final Model Predictions and Comparison with True Prices

In [ ]:
best_global_model

In [ ]:
# Final Prediction Model
final_model = best_global_model

# Make predictions on the test set using the final model
y_final_pred = final_model.predict(X_test)
final_y_pred = (y_final_pred)
final_y_test =(y_test)

In [ ]:
# Create a DataFrame with the predicted prices and true prices
results = pd.DataFrame({'Predicted Price': final_y_pred, 'True Price': final_y_test})

# Calculate the difference between the true prices and predicted prices and add a new column
results['Difference'] = results['True Price'] - results['Predicted Price']

# Display the results
print(results)